In [100]:
# Plot the uncorrected torsion time-series from the Messung24 dataset


%reset -f
%matplotlib qt

import glob

import easygui
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

import scipy.io

plt.close("all")
plt.style.use('default')

START_FRAME = 0
END_FRAME = 25000

base = "Messung24/woVicPy_woYawCorr/Torsion"  #easygui.diropenbox("Select the folder containing the /Tor_* folders:")
paths = glob.glob(base + "/Tor_*")

yaw_angle_file = "Messung24/Messung24_yaw.mat"

yaw_angle_data = scipy.io.loadmat(yaw_angle_file)
yaw_angle_data = yaw_angle_data["yaw_wea"].flatten()
yaw_angle_data = yaw_angle_data - yaw_angle_data[0]

assert len(paths) > 0

names = [i for i in range(len(paths))]

# ----------------------------------
# blender_data = pd.read_csv("sandbox/BLENDER Blade BLADE_A_1.csv", sep=";", decimal=",")

real_angle = None  #blender_data[" Winkel"] - blender_data[" Winkel"].iloc[0]

# ----------------------------------------

diff_list = []
point_pair_diff_list = []
angles = []

for i in range(len(paths)):

    file_1 = paths[i] + "/Blade_0_AOI_2_P1.csv"
    file_2 = paths[i] + "/Blade_0_AOI_2_P2.csv"

    df1 = pd.read_csv(file_1, sep=";", decimal=',', skiprows=1, header=0)
    df2 = pd.read_csv(file_2, sep=";", decimal=',', skiprows=1, header=0)

    sigma_1 = df1["sigma [pixel]"].to_numpy()
    sigma_2 = df2["sigma [pixel]"].to_numpy()
    print(file_1)

    offset1 = np.array([df1["X [mm]"].to_numpy(), df1["Y [mm]"].to_numpy(), df1["Z [mm]"].to_numpy()]).T
    offset2 = np.array([df2["X [mm]"].to_numpy(), df2["Y [mm]"].to_numpy(), df2["Z [mm]"].to_numpy()]).T

    change1 = np.array([df1["U [mm]"].to_numpy(), df1["V [mm]"].to_numpy(), df1["W [mm]"].to_numpy()]).T
    change2 = np.array([df2["U [mm]"].to_numpy(), df2["V [mm]"].to_numpy(), df2["W [mm]"].to_numpy()]).T

    points1 = offset1 + change1
    points2 = offset2 + change2

    diff = points1 - points2

    bad_sigmas = np.where((sigma_1 < 0) | (sigma_2 < 0))[0]

    distance = np.linalg.norm(diff, axis=1)

    angle = np.arcsin((diff.T[0] / distance))
    angle = np.rad2deg(angle)
    angle = angle - angle[0]

    if real_angle is None:
        real_angle = np.full(len(sigma_1), 0)

    angle[bad_sigmas] = 0
    if len(angle) == len(real_angle):
        angle_diff = angle - real_angle
        angle_diff[bad_sigmas] = np.nan
        diff_list.append(angle_diff)

    # point_pair_diff_list.append(diff_x)
    other_diff = np.linalg.norm(diff.T[0:3], axis=0)
    other_diff_relativ = ((other_diff - other_diff[0]) * 1000) / other_diff[0]
    other_diff_relativ[bad_sigmas] = 0
    other_diff_relativ[bad_sigmas] = np.max(other_diff_relativ)
    point_pair_diff_list.append(other_diff_relativ)
    angle[bad_sigmas] = np.nan
    angles.append(angle)
    #plt.plot(angle, label=names[i])

if len(diff_list) > 0:
    for i in range(len(diff_list)):
        print(names[i], ": Mean: ", np.nanmean(np.abs(diff_list[i])), "  Sdt: ", np.std(np.abs(diff_list[i])),
              "  Max: ", np.max(np.abs(diff_list[i])))

point_pair_diff_list = np.array(point_pair_diff_list)

mean_point_pair_diff = np.nanmean(np.abs(point_pair_diff_list), axis=1)

better_half = np.sort(mean_point_pair_diff)[len(mean_point_pair_diff) // 2]
good_values = np.where(mean_point_pair_diff < better_half)[0]

angles = np.array(angles)
angle_mean = np.nanmean(angles[good_values], axis=0)

fig, (ax, ax1) = plt.subplots(2, 1, sharex=True)

ax.plot(angle_mean)
ax.set_ylabel("Average Torsion [°]")
ax.grid(True)
ax.set_xlim(START_FRAME, END_FRAME)
plt.tight_layout()

ax1.plot(yaw_angle_data)
ax1.set_xlabel("Frame No.")
ax1.set_ylabel("Yaw Angle Offset [°]")
ax1.grid(True)
ax1.set_xlim(START_FRAME, END_FRAME)
ax1.set_ylim(START_FRAME)
fig.align_ylabels()

plt.savefig('../../graphics/yaw_problem_statement.pdf', format='pdf', transparent=True, bbox_inches='tight')


Messung24/woVicPy_woYawCorr/Torsion/Tor_2/Blade_0_AOI_2_P1.csv
Messung24/woVicPy_woYawCorr/Torsion/Tor_20/Blade_0_AOI_2_P1.csv
Messung24/woVicPy_woYawCorr/Torsion/Tor_9/Blade_0_AOI_2_P1.csv
Messung24/woVicPy_woYawCorr/Torsion/Tor_16/Blade_0_AOI_2_P1.csv
Messung24/woVicPy_woYawCorr/Torsion/Tor_7/Blade_0_AOI_2_P1.csv
Messung24/woVicPy_woYawCorr/Torsion/Tor_3/Blade_0_AOI_2_P1.csv
Messung24/woVicPy_woYawCorr/Torsion/Tor_6/Blade_0_AOI_2_P1.csv
Messung24/woVicPy_woYawCorr/Torsion/Tor_24/Blade_0_AOI_2_P1.csv
Messung24/woVicPy_woYawCorr/Torsion/Tor_4/Blade_0_AOI_2_P1.csv
Messung24/woVicPy_woYawCorr/Torsion/Tor_14/Blade_0_AOI_2_P1.csv
Messung24/woVicPy_woYawCorr/Torsion/Tor_11/Blade_0_AOI_2_P1.csv
Messung24/woVicPy_woYawCorr/Torsion/Tor_23/Blade_0_AOI_2_P1.csv
Messung24/woVicPy_woYawCorr/Torsion/Tor_10/Blade_0_AOI_2_P1.csv
Messung24/woVicPy_woYawCorr/Torsion/Tor_0/Blade_0_AOI_2_P1.csv
Messung24/woVicPy_woYawCorr/Torsion/Tor_13/Blade_0_AOI_2_P1.csv
Messung24/woVicPy_woYawCorr/Torsion/Tor_17/Blad

In [107]:
# Plot the uncorrected torsion time-series from the Messung24 dataset


%reset -f
%matplotlib qt

import glob

import easygui
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

import scipy.io

plt.close("all")
plt.style.use('default')

START_FRAME = 0
END_FRAME = 25000

base_woyawcorr = "Messung24/woVicPy_woYawCorr/Torsion"
base_wyawcorr = "Messung24/woVicPy_wYawCorr/Torsion"
base_list = [base_woyawcorr, base_wyawcorr]
fig, (ax, ax1) = plt.subplots(2, 1, sharex=True)
for base in base_list:
    paths = glob.glob(base + "/Tor_*")

    yaw_angle_file = "Messung24/Messung24_yaw.mat"

    yaw_angle_data = scipy.io.loadmat(yaw_angle_file)
    yaw_angle_data = yaw_angle_data["yaw_wea"].flatten()
    yaw_angle_data = yaw_angle_data - yaw_angle_data[0]

    assert len(paths) > 0

    names = [i for i in range(len(paths))]

    # ----------------------------------
    # blender_data = pd.read_csv("sandbox/BLENDER Blade BLADE_A_1.csv", sep=";", decimal=",")

    real_angle = None  #blender_data[" Winkel"] - blender_data[" Winkel"].iloc[0]

    # ----------------------------------------

    diff_list = []
    point_pair_diff_list = []
    angles = []

    for i in range(len(paths)):

        file_1 = paths[i] + "/Blade_0_AOI_2_P1.csv"
        file_2 = paths[i] + "/Blade_0_AOI_2_P2.csv"

        df1 = pd.read_csv(file_1, sep=";", decimal=',', skiprows=1, header=0)
        df2 = pd.read_csv(file_2, sep=";", decimal=',', skiprows=1, header=0)

        sigma_1 = df1["sigma [pixel]"].to_numpy()
        sigma_2 = df2["sigma [pixel]"].to_numpy()
        print(file_1)

        offset1 = np.array([df1["X [mm]"].to_numpy(), df1["Y [mm]"].to_numpy(), df1["Z [mm]"].to_numpy()]).T
        offset2 = np.array([df2["X [mm]"].to_numpy(), df2["Y [mm]"].to_numpy(), df2["Z [mm]"].to_numpy()]).T

        change1 = np.array([df1["U [mm]"].to_numpy(), df1["V [mm]"].to_numpy(), df1["W [mm]"].to_numpy()]).T
        change2 = np.array([df2["U [mm]"].to_numpy(), df2["V [mm]"].to_numpy(), df2["W [mm]"].to_numpy()]).T

        points1 = offset1 + change1
        points2 = offset2 + change2

        diff = points1 - points2

        bad_sigmas = np.where((sigma_1 < 0) | (sigma_2 < 0))[0]

        distance = np.linalg.norm(diff, axis=1)

        angle = np.arcsin((diff.T[0] / distance))
        angle = np.rad2deg(angle)
        angle = angle - angle[0]

        if real_angle is None:
            real_angle = np.full(len(sigma_1), 0)

        angle[bad_sigmas] = 0
        if len(angle) == len(real_angle):
            angle_diff = angle - real_angle
            angle_diff[bad_sigmas] = np.nan
            diff_list.append(angle_diff)

        # point_pair_diff_list.append(diff_x)
        other_diff = np.linalg.norm(diff.T[0:3], axis=0)
        other_diff_relativ = ((other_diff - other_diff[0]) * 1000) / other_diff[0]
        other_diff_relativ[bad_sigmas] = 0
        other_diff_relativ[bad_sigmas] = np.max(other_diff_relativ)
        point_pair_diff_list.append(other_diff_relativ)
        angle[bad_sigmas] = np.nan
        angles.append(angle)
        #plt.plot(angle, label=names[i])

    if len(diff_list) > 0:
        for i in range(len(diff_list)):
            print(names[i], ": Mean: ", np.nanmean(np.abs(diff_list[i])), "  Sdt: ", np.std(np.abs(diff_list[i])),
                  "  Max: ", np.max(np.abs(diff_list[i])))

    point_pair_diff_list = np.array(point_pair_diff_list)

    mean_point_pair_diff = np.nanmean(np.abs(point_pair_diff_list), axis=1)

    better_half = np.sort(mean_point_pair_diff)[len(mean_point_pair_diff) // 2]
    good_values = np.where(mean_point_pair_diff < better_half)[0]

    angles = np.array(angles)
    angle_mean = np.nanmean(angles[good_values], axis=0)

    ax.plot(angle_mean)
ax.set_ylabel("Average Torsion [°]")
ax.grid(True)
ax.set_xlim(START_FRAME, END_FRAME)
ax.legend(["w/o Yaw Correction","w/ Yaw Correction"], loc="lower left")
plt.tight_layout()

ax1.plot(yaw_angle_data)
ax1.set_xlabel("Frame No.")
ax1.set_ylabel("Yaw Angle Offset [°]")
ax1.grid(True)
ax1.set_xlim(START_FRAME, END_FRAME)
ax1.set_ylim(START_FRAME)
fig.align_ylabels()

plt.savefig('../../graphics/yaw_messung24.pdf', format='pdf', transparent=True, bbox_inches='tight')


Messung24/woVicPy_woYawCorr/Torsion/Tor_2/Blade_0_AOI_2_P1.csv
Messung24/woVicPy_woYawCorr/Torsion/Tor_20/Blade_0_AOI_2_P1.csv
Messung24/woVicPy_woYawCorr/Torsion/Tor_9/Blade_0_AOI_2_P1.csv
Messung24/woVicPy_woYawCorr/Torsion/Tor_16/Blade_0_AOI_2_P1.csv
Messung24/woVicPy_woYawCorr/Torsion/Tor_7/Blade_0_AOI_2_P1.csv
Messung24/woVicPy_woYawCorr/Torsion/Tor_3/Blade_0_AOI_2_P1.csv
Messung24/woVicPy_woYawCorr/Torsion/Tor_6/Blade_0_AOI_2_P1.csv
Messung24/woVicPy_woYawCorr/Torsion/Tor_24/Blade_0_AOI_2_P1.csv
Messung24/woVicPy_woYawCorr/Torsion/Tor_4/Blade_0_AOI_2_P1.csv
Messung24/woVicPy_woYawCorr/Torsion/Tor_14/Blade_0_AOI_2_P1.csv
Messung24/woVicPy_woYawCorr/Torsion/Tor_11/Blade_0_AOI_2_P1.csv
Messung24/woVicPy_woYawCorr/Torsion/Tor_23/Blade_0_AOI_2_P1.csv
Messung24/woVicPy_woYawCorr/Torsion/Tor_10/Blade_0_AOI_2_P1.csv
Messung24/woVicPy_woYawCorr/Torsion/Tor_0/Blade_0_AOI_2_P1.csv
Messung24/woVicPy_woYawCorr/Torsion/Tor_13/Blade_0_AOI_2_P1.csv
Messung24/woVicPy_woYawCorr/Torsion/Tor_17/Blad

In [102]:
# SYNTHETIC

%reset -f
%matplotlib qt

import glob

import easygui
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

import scipy.io

PLOT_SCALE_FACTOR = 1.5
plt.close("all")
plt.style.use('default')

START_FRAME = 0
END_FRAME = 865

base_woyawcorr = "Blender_Gier/Torsion_woYawCorr"
base_wyawcorr = "Blender_Gier/Torsion_wYawCorr"
base_list = [base_woyawcorr, base_wyawcorr]

blender_baseline = -(np.loadtxt("Blender_Gier/torsion_Gier.csv", delimiter=',', skiprows=1) - 90)[
    START_FRAME:END_FRAME + 1]

yaw_angle_data = np.loadtxt("Blender_Gier/rotor_center_z_rotation.csv", delimiter=',', skiprows=1)
yaw_angle_data = yaw_angle_data[:, 1] - yaw_angle_data[0, 0]

fig, (ax, ax2, ax1) = plt.subplots(3, 1, sharex=True)
ax.plot(blender_baseline, color="tab:green")
for base in base_list[:]:
    paths = glob.glob(base + "/Tor_*")

    assert len(paths) > 0

    names = [i for i in range(len(paths))]

    # ----------------------------------
    # blender_data = pd.read_csv("sandbox/BLENDER Blade BLADE_A_1.csv", sep=";", decimal=",")

    real_angle = None  #blender_data[" Winkel"] - blender_data[" Winkel"].iloc[0]

    # ----------------------------------------

    diff_list = []
    point_pair_diff_list = []
    angles = []

    for i in range(len(paths)):

        file_1 = paths[i] + "/Blade_0_AOI_2_P1.csv"
        file_2 = paths[i] + "/Blade_0_AOI_2_P2.csv"

        df1 = pd.read_csv(file_1, sep=";", decimal=',', skiprows=1, header=0)
        df2 = pd.read_csv(file_2, sep=";", decimal=',', skiprows=1, header=0)

        sigma_1 = df1["sigma [pixel]"].to_numpy()
        sigma_2 = df2["sigma [pixel]"].to_numpy()
        print(file_1)

        offset1 = np.array([df1["X [mm]"].to_numpy(), df1["Y [mm]"].to_numpy(), df1["Z [mm]"].to_numpy()]).T
        offset2 = np.array([df2["X [mm]"].to_numpy(), df2["Y [mm]"].to_numpy(), df2["Z [mm]"].to_numpy()]).T

        change1 = np.array([df1["U [mm]"].to_numpy(), df1["V [mm]"].to_numpy(), df1["W [mm]"].to_numpy()]).T
        change2 = np.array([df2["U [mm]"].to_numpy(), df2["V [mm]"].to_numpy(), df2["W [mm]"].to_numpy()]).T

        points1 = offset1 + change1
        points2 = offset2 + change2

        diff = points1 - points2

        bad_sigmas = np.where((sigma_1 < 0) | (sigma_2 < 0))[0]

        distance = np.linalg.norm(diff, axis=1)

        angle = np.arcsin((diff.T[0] / distance))
        angle = np.rad2deg(angle)
        angle = angle - angle[0]

        if real_angle is None:
            real_angle = np.full(len(sigma_1), 0)

        angle[bad_sigmas] = 0
        if len(angle) == len(real_angle):
            angle_diff = angle - real_angle
            angle_diff[bad_sigmas] = np.nan
            diff_list.append(angle_diff)

        # point_pair_diff_list.append(diff_x)
        other_diff = np.linalg.norm(diff.T[0:3], axis=0)
        other_diff_relativ = ((other_diff - other_diff[0]) * 1000) / other_diff[0]
        other_diff_relativ[bad_sigmas] = 0
        other_diff_relativ[bad_sigmas] = np.max(other_diff_relativ)
        point_pair_diff_list.append(other_diff_relativ)
        angle[bad_sigmas] = np.nan
        angles.append(angle)
        #plt.plot(angle, label=names[i])

    if len(diff_list) > 0:
        for i in range(len(diff_list)):
            print(names[i], ": Mean: ", np.nanmean(np.abs(diff_list[i])), "  Sdt: ", np.std(np.abs(diff_list[i])),
                  "  Max: ", np.max(np.abs(diff_list[i])))

    point_pair_diff_list = np.array(point_pair_diff_list)

    mean_point_pair_diff = np.nanmean(np.abs(point_pair_diff_list), axis=1)

    better_half = np.sort(mean_point_pair_diff)[len(mean_point_pair_diff) // 2]
    good_values = np.where(mean_point_pair_diff < better_half)[0]

    angles = np.array(angles)
    angle_mean = np.nanmean(angles[good_values], axis=0)

    ax.plot(angle_mean)

    absolute_error = angle_mean - blender_baseline

    ax2.plot(absolute_error)

ax.set_ylabel("Average Torsion [°]")
ax.set_xlim(START_FRAME, END_FRAME)
ax.legend(["Blender Baseline", "w/o Yaw Correction", "w/ Yaw Correction"], loc="lower left")
ax.grid(True)

ax2.grid(True)
ax2.set_ylabel("Absolute Error [°]")

ax1.plot(yaw_angle_data[START_FRAME:END_FRAME])
ax1.set_xlabel("Frame No.")
ax1.set_ylabel("Yaw Angle Offset [°]")
ax1.grid(True)
ax1.set_xlim(START_FRAME, END_FRAME)
fig.align_ylabels()
plt.tight_layout()

fig.set_size_inches(PLOT_SCALE_FACTOR * np.array((6.4, 4.8)))
plt.savefig('../../graphics/yaw_blender.pdf', format='pdf', transparent=True, bbox_inches='tight')
plt.show()

Blender_Gier/Torsion_woYawCorr/Tor_2/Blade_0_AOI_2_P1.csv
Blender_Gier/Torsion_woYawCorr/Tor_9/Blade_0_AOI_2_P1.csv
Blender_Gier/Torsion_woYawCorr/Tor_7/Blade_0_AOI_2_P1.csv
Blender_Gier/Torsion_woYawCorr/Tor_3/Blade_0_AOI_2_P1.csv
Blender_Gier/Torsion_woYawCorr/Tor_6/Blade_0_AOI_2_P1.csv
Blender_Gier/Torsion_woYawCorr/Tor_4/Blade_0_AOI_2_P1.csv
Blender_Gier/Torsion_woYawCorr/Tor_14/Blade_0_AOI_2_P1.csv
Blender_Gier/Torsion_woYawCorr/Tor_11/Blade_0_AOI_2_P1.csv
Blender_Gier/Torsion_woYawCorr/Tor_10/Blade_0_AOI_2_P1.csv
Blender_Gier/Torsion_woYawCorr/Tor_0/Blade_0_AOI_2_P1.csv
Blender_Gier/Torsion_woYawCorr/Tor_13/Blade_0_AOI_2_P1.csv
Blender_Gier/Torsion_woYawCorr/Tor_1/Blade_0_AOI_2_P1.csv
Blender_Gier/Torsion_woYawCorr/Tor_12/Blade_0_AOI_2_P1.csv
Blender_Gier/Torsion_woYawCorr/Tor_8/Blade_0_AOI_2_P1.csv
Blender_Gier/Torsion_woYawCorr/Tor_5/Blade_0_AOI_2_P1.csv
0 : Mean:  4.2021046777401265   Sdt:  2.536626421488144   Max:  10.356680249399803
1 : Mean:  4.124531303452086   Sdt:  2.536

In [105]:
# Plot error bars

%reset -f
%matplotlib qt
import glob
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import medfilt

# Plot deformation

plt.close("all")
plt.style.use('default')
PLOT_SCALE_FACTOR = 1

START_FRAME = 0
END_FRAME = 865

base_woyawcorr = "Blender_Gier/Torsion_woYawCorr"
base_wyawcorr = "Blender_Gier/Torsion_wYawCorr"
base_list = [base_woyawcorr, base_wyawcorr]

blender_baseline = -(np.loadtxt("Blender_Gier/torsion_Gier.csv", delimiter=',', skiprows=1) - 90)[
    START_FRAME:END_FRAME + 1]

yaw_angle_data = np.loadtxt("Blender_Gier/rotor_center_z_rotation.csv", delimiter=',', skiprows=1)
yaw_angle_data = yaw_angle_data[:, 1] - yaw_angle_data[0, 0]

bin_centers = np.round(np.linspace(blender_baseline.min(), blender_baseline.max(), 10), 2)
bin_width = bin_centers[1] - bin_centers[0]
bar_width = 0.125

fig1, ax3 = plt.subplots(1, 1)

# Use 'i' for the outer loop (0 or 1), keep it safe!
for i, base in enumerate(base_list):
    paths = glob.glob(base + "/Tor_*")

    assert len(paths) > 0

    names = [j for j in range(len(paths))]

    diff_list = []
    point_pair_diff_list = []
    angles = []

    # Rename inner loop variable from 'i' to 'j' to avoid overwriting outer 'i'
    for j in range(len(paths)):

        file_1 = paths[j] + "/Blade_0_AOI_2_P1.csv"
        file_2 = paths[j] + "/Blade_0_AOI_2_P2.csv"

        df1 = pd.read_csv(file_1, sep=";", decimal=',', skiprows=1, header=0)
        df2 = pd.read_csv(file_2, sep=";", decimal=',', skiprows=1, header=0)

        sigma_1 = df1["sigma [pixel]"].to_numpy()
        sigma_2 = df2["sigma [pixel]"].to_numpy()

        offset1 = np.array([df1["X [mm]"].to_numpy(), df1["Y [mm]"].to_numpy(), df1["Z [mm]"].to_numpy()]).T
        offset2 = np.array([df2["X [mm]"].to_numpy(), df2["Y [mm]"].to_numpy(), df2["Z [mm]"].to_numpy()]).T

        change1 = np.array([df1["U [mm]"].to_numpy(), df1["V [mm]"].to_numpy(), df1["W [mm]"].to_numpy()]).T
        change2 = np.array([df2["U [mm]"].to_numpy(), df2["V [mm]"].to_numpy(), df2["W [mm]"].to_numpy()]).T

        points1 = offset1 + change1
        points2 = offset2 + change2

        diff = points1 - points2

        bad_sigmas = np.where((sigma_1 < 0) | (sigma_2 < 0))[0]

        distance = np.linalg.norm(diff, axis=1)

        angle = np.arcsin((diff.T[0] / distance))
        angle = np.rad2deg(angle)
        angle = angle - angle[0]

        real_angle = np.full(len(sigma_1), 0)

        angle[bad_sigmas] = 0
        if len(angle) == len(real_angle):
            angle_diff = angle - real_angle
            angle_diff[bad_sigmas] = np.nan
            diff_list.append(angle_diff)

        other_diff = np.linalg.norm(diff.T[0:3], axis=0)
        other_diff_relativ = ((other_diff - other_diff[0]) * 1000) / other_diff[0]
        other_diff_relativ[bad_sigmas] = 0
        other_diff_relativ[bad_sigmas] = np.max(other_diff_relativ)
        point_pair_diff_list.append(other_diff_relativ)
        angle[bad_sigmas] = np.nan
        angles.append(angle)

    if len(diff_list) > 0:
        for j in range(len(diff_list)):
            print(names[j], ": Mean: ", np.nanmean(np.abs(diff_list[j])), "  Sdt: ", np.std(np.abs(diff_list[j])),
                  "  Max: ", np.max(np.abs(diff_list[j])))

    point_pair_diff_list = np.array(point_pair_diff_list)

    mean_point_pair_diff = np.nanmean(np.abs(point_pair_diff_list), axis=1)

    better_half = np.sort(mean_point_pair_diff)[len(mean_point_pair_diff) // 2]
    good_values = np.where(mean_point_pair_diff < better_half)[0]

    angles = np.array(angles)
    angle_mean = np.nanmean(angles[good_values], axis=0)

    absolute_error = angle_mean - blender_baseline

    binned_means = []
    binned_errors = []
    valid_bin_centers = []

    for center in bin_centers:
        mask = (angle_mean >= center - bin_width / 2) & \
               (angle_mean < center + bin_width / 2)
        if np.any(mask):
            subset_errors = absolute_error[mask]
            binned_means.append(np.mean(subset_errors))

            n_points = len(subset_errors)
            sem = np.std(subset_errors, ddof=1) / np.sqrt(n_points) if n_points > 1 else 0.0
            binned_errors.append(sem)
            valid_bin_centers.append(center)

    offset = (i - 0.5) * bar_width
    label_text = "w/o Yaw Correction" if i == 0 else "w/ Yaw Correction"

    # Plot using valid_bin_centers to guarantee matching shapes
    ax3.bar(
        np.array(valid_bin_centers) + offset,
        binned_means,
        width=bar_width,
        linewidth=0.8,
        edgecolor="black",
        yerr=binned_errors,
        capsize=3,
        error_kw={'elinewidth': 1, 'ecolor': 'black'},
        label=label_text
    )

    MAE = np.abs(absolute_error).mean()
    AE_max = np.abs(absolute_error).max()
    SEM = np.abs(absolute_error).std() / np.sqrt(len(absolute_error))

    print(f"{label_text}")
    print(f"MAE= {MAE} °")
    print(f"MAximum = {AE_max} °")
    print(f"SEM = {SEM} °")
    print("----------------------------------------------------------")

ax3.axhline(0, color='black', linewidth=0.8)
ax3.set_xlabel("Average Torsion [°]")
ax3.set_ylabel("Mean Absolute Error [°]")
ax3.set_xticks(bin_centers)
ax3.grid(True, linestyle='-', alpha=0.5, color='gray')
ax3.legend()
fig1.tight_layout()
fig1.set_size_inches(PLOT_SCALE_FACTOR * np.array((6.4, 4.8)))
fig1.savefig('../../graphics/yaw_error_bars_binned.pdf', format='pdf', bbox_inches='tight')

plt.show()

0 : Mean:  4.2021046777401265   Sdt:  2.536626421488144   Max:  10.356680249399803
1 : Mean:  4.124531303452086   Sdt:  2.536288860453258   Max:  10.349186442688033
2 : Mean:  4.212268331382765   Sdt:  nan   Max:  nan
3 : Mean:  4.185090148080846   Sdt:  2.5319474469020027   Max:  10.391276053557092
4 : Mean:  4.153009507402738   Sdt:  2.552743560425982   Max:  10.340816765127151
5 : Mean:  4.1606950918460495   Sdt:  2.534850913626045   Max:  10.355072199311293
6 : Mean:  4.1105585406759655   Sdt:  2.5462579152353633   Max:  10.320649406480756
7 : Mean:  4.175682525184731   Sdt:  2.54473315653312   Max:  10.31851120507138
8 : Mean:  4.200597435148892   Sdt:  2.546027267584802   Max:  10.401010511466295
9 : Mean:  4.178909658504423   Sdt:  2.5358998537377295   Max:  10.332785769483953
10 : Mean:  4.157783217436114   Sdt:  2.5483622288442898   Max:  10.330419841048244
11 : Mean:  4.25344777858494   Sdt:  2.544209681220885   Max:  10.460617362034832
12 : Mean:  4.178186264763984   Sdt:  2